# LeetCode 3934 — Smallest Unique Subarray bằng Python

- Link: https://leetcode.com/problems/smallest-unique-subarray/
- Độ khó: **Hard**
- Chủ đề: Xử lý chuỗi/mảng, tìm kiếm nhị phân trên đáp án, băm cuộn (rolling hash), mảng hậu tố (suffix array) + LCP, automat hậu tố (suffix automaton)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def smallestUniqueSubarray(self, nums: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums`. Một mảng con (subarray) gọi là **duy nhất** nếu **không có mảng con nào khác (ở vị trí khác)** giống hệt nó (cùng độ dài, cùng các phần tử). Tìm **độ dài nhỏ nhất** của một mảng con duy nhất.

Ví dụ `[2,1,2,3,3]`: mảng con `[1]` chỉ xuất hiện một lần ⇒ đáp án `1`. Với `[3,3,3]` thì `[3]` xuất hiện 3 lần, `[3,3]` xuất hiện 2 lần (chồng lấn), chỉ cả mảng là duy nhất ⇒ `3`.

Bài này hay vì có **nhiều cách chính đáng** với độ khó tăng dần: từ vét cạn `O(n³)`, qua tìm nhị phân + băm cuộn `O(n log n)`, tới cấu trúc dữ liệu chuỗi "cao cấp" (suffix array, suffix automaton) cho đáp án **chính xác, không xác suất**.

Sau notebook này bạn sẽ:

1. Chứng minh **tính đơn điệu**: nếu có mảng con duy nhất độ dài `L` (và `L < n`) thì cũng có mảng con duy nhất độ dài `L + 1` ⇒ tìm nhị phân được.
2. Cài băm cuộn `hash(window) = (H[i+L] − H[i]·B^L) mod P` và hiểu **rủi ro va chạm** (collision).
3. Biết `LCP` (longest common prefix) của hai hậu tố kề nhau trong mảng hậu tố cho **độ dài ngắn nhất để một hậu tố trở nên duy nhất**.
4. Biết automat hậu tố: trạng thái nào có đúng **một vị trí kết thúc** cho đáp án `len(link) + 1`.
5. Viết 4 lời giải (vét cạn, nhị phân + băm, mảng hậu tố + LCP, automat hậu tố) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums`. Tìm độ dài nhỏ nhất của một mảng con (các phần tử liên tiếp) **không giống hệt bất kỳ mảng con nào khác** trong `nums`. Hai mảng con giống nhau nếu cùng độ dài và cùng phần tử ở từng vị trí; hai mảng con đếm là khác nhau nếu **bắt đầu ở chỉ số khác nhau** (kể cả chồng lấn).

**Ví dụ 1:** `nums = [3,3,3]` → `3`.

**Ví dụ 2:** `nums = [2,1,2,3,3]` → `1` (`[1]` duy nhất).

**Ví dụ 3:** `nums = [1,1,2,2,1]` → `2` (không có phần tử đơn nào duy nhất; độ dài 2 thì `[1,1]`, `[1,2]`, `[2,2]`, `[2,1]` đều duy nhất).

**Ràng buộc:** `1 ≤ len(nums) ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`.

In [1]:
import random, time, math
from collections import Counter

EXAMPLES = [
    ([3, 3, 3], 3),
    ([2, 1, 2, 3, 3], 1),
    ([1, 1, 2, 2, 1], 2),
]
for nums, exp in EXAMPLES:
    print(nums, "-> kỳ vọng", exp)

[3, 3, 3] -> kỳ vọng 3
[2, 1, 2, 3, 3] -> kỳ vọng 1
[1, 1, 2, 2, 1] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Cửa sổ trượt và khóa `tuple`

Các mảng con độ dài `L` là các "cửa sổ" `nums[i:i+L]` với `i = 0..n−L`. Để đếm số lần xuất hiện ta cần khóa **băm được**: `list` không băm được, `tuple` thì được.

In [2]:
nums = [1, 1, 2, 2, 1]
L = 2
windows = [tuple(nums[i:i + L]) for i in range(len(nums) - L + 1)]
print(windows)
print(Counter(windows))
try:
    {[1, 2]: 1}
except TypeError as e:
    print("list không làm khóa được:", e)

[(1, 1), (1, 2), (2, 2), (2, 1)]
Counter({(1, 1): 1, (1, 2): 1, (2, 2): 1, (2, 1): 1})
list không làm khóa được: unhashable type: 'list'


### 3.2. Tìm nhị phân trên đáp án

Nếu một điều kiện `P(L)` là `False ... False True ... True` (đơn điệu), ta tìm `L` nhỏ nhất thỏa `P` bằng nhị phân trong `O(log n)` lần kiểm tra:

```
lo, hi = 1, n
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid
    else: lo = mid + 1
```

In [3]:
def first_true(lo, hi, P):
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(1, 100, lambda L: L * L >= 50))     # 8

8


### 3.3. Băm cuộn đa thức (polynomial rolling hash)

Gán cho mảng `a₀, a₁, ..., a_{L−1}` giá trị `h = a₀·B^{L−1} + a₁·B^{L−2} + ... + a_{L−1}` modulo một số nguyên tố lớn `P`. Với tiền tố `H[i] = hash(nums[:i])` ta có `H[i+1] = H[i]·B + nums[i]`, và băm của cửa sổ `[i, i+L)` là

```
hash = (H[i+L] − H[i] · B^L) mod P
```

Mỗi cửa sổ tính trong `O(1)` sau khi chuẩn bị `H` và lũy thừa `B^L`. Hai cửa sổ giống hệt nhau luôn cho cùng băm; hai cửa sổ khác nhau có thể trùng băm (**va chạm**) với xác suất nhỏ.

In [4]:
P = (1 << 61) - 1
B = 911382323

def prefix_hash(nums, B=B, P=P):
    H = [0]
    for x in nums:
        H.append((H[-1] * B + x) % P)
    return H

nums = [1, 1, 2, 2, 1]
H = prefix_hash(nums)
L = 2
BL = pow(B, L, P)
hs = [(H[i + L] - H[i] * BL) % P for i in range(len(nums) - L + 1)]
direct = [(nums[i] * B + nums[i + 1]) % P for i in range(len(nums) - 1)]
assert hs == direct
print("băm cửa sổ độ dài 2 khớp công thức trực tiếp:", len(set(hs)), "giá trị khác nhau")

băm cửa sổ độ dài 2 khớp công thức trực tiếp: 4 giá trị khác nhau


### 3.4. Xác suất va chạm (vì sao không dùng `mod = 10^9 + 7` đơn lẻ)

Với `n` cửa sổ và mô-đun `P`, số cặp là `≈ n²/2`; mỗi cặp khác nhau trùng băm với xác suất cỡ `1/P`. Kỳ vọng số va chạm `≈ n² / (2P)`. Với `n = 10^5` và `P = 10^9 + 7` ta có khoảng **5 va chạm**; với `P = 2^61 − 1` chỉ khoảng `2·10^-9`.

In [5]:
n = 10**5
for P_ in (10**9 + 7, (1 << 61) - 1):
    print(f"P = {P_}: kỳ vọng va chạm ≈ {n * n / (2 * P_):.3g}")

P = 1000000007: kỳ vọng va chạm ≈ 5
P = 2305843009213693951: kỳ vọng va chạm ≈ 2.17e-09


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: `nums = [1,1,2,2,1]`.

| Độ dài `L` | Các cửa sổ | Có cửa sổ xuất hiện đúng 1 lần? |
|---|---|---|
| 1 | `[1]×3`, `[2]×2` | không |
| 2 | `[1,1]`, `[1,2]`, `[2,2]`, `[2,1]` (mỗi cái 1 lần) | **có** |
| 3 | `[1,1,2]`, `[1,2,2]`, `[2,2,1]` | có |
| 4, 5 | đều duy nhất | có |

Đáp án = `L` nhỏ nhất có cửa sổ duy nhất = `2`.

**Quan sát đơn điệu.** Nếu một cửa sổ độ dài `L` là duy nhất thì **mở rộng nó thêm một phần tử** (sang phải, hoặc sang trái nếu đã chạm cuối mảng) cho cửa sổ độ dài `L + 1` cũng duy nhất. Vì sao? Bất kỳ lần xuất hiện thứ hai của cửa sổ dài sẽ chứa một lần xuất hiện thứ hai của cửa sổ ngắn (tại vị trí khác), mâu thuẫn. Do đó, bảng "có cửa sổ duy nhất không?" theo `L` có dạng `không, không, ..., có, có, ...` (và `L = n` luôn là `có`).

In [6]:
def has_unique_window_exact(nums, L):
    c = Counter(tuple(nums[i:i + L]) for i in range(len(nums) - L + 1))
    return 1 in c.values()

for nums, exp in EXAMPLES:
    print(nums, [has_unique_window_exact(nums, L) for L in range(1, len(nums) + 1)])
# kiểm chứng tính đơn điệu trên nhiều mảng ngẫu nhiên
rng = random.Random(0)
for _ in range(500):
    a = [rng.randint(1, 3) for _ in range(rng.randint(1, 12))]
    flags = [has_unique_window_exact(a, L) for L in range(1, len(a) + 1)]
    assert flags == sorted(flags) and flags[-1] is True, a
print("tính đơn điệu đúng trên 500 mảng ngẫu nhiên")

[3, 3, 3] [False, False, True]
[2, 1, 2, 3, 3] [True, True, True, True, True]
[1, 1, 2, 2, 1] [False, True, True, True, True]
tính đơn điệu đúng trên 500 mảng ngẫu nhiên


## 5. Cách 1 — Brute Force

**Ý tưởng.** Với `L = 1, 2, ...`: đếm mọi cửa sổ độ dài `L` bằng `Counter` các `tuple`; trả về `L` đầu tiên có giá trị đếm `1`.

In [7]:
def brute(nums):
    n = len(nums)
    for L in range(1, n + 1):
        if has_unique_window_exact(nums, L):
            return L

for nums, exp in EXAMPLES:
    got = brute(nums)
    print(nums, "->", got)
    assert got == exp

[3, 3, 3] -> 3
[2, 1, 2, 3, 3] -> 1
[1, 1, 2, 2, 1] -> 2


**Độ phức tạp.** Với mỗi `L`: `n − L + 1` cửa sổ, mỗi cửa sổ tốn `O(L)` để tạo tuple và băm ⇒ `O(n·L)`; cộng qua `L` đến đáp án: `O(n · ans²)` cho đến `O(n³)` khi đáp án `≈ n` (như mảng toàn số giống nhau). Không gian `O(n·L)`.

**Nút thắt.** (1) Thử từng `L` tuần tự; (2) so sánh/băm cửa sổ tốn `O(L)`. Giải quyết (1) bằng tìm nhị phân, (2) bằng băm cuộn.

## 6. Bước đột phá

**Đột phá 1 — tìm nhị phân.** Vì `P(L) = "tồn tại cửa sổ duy nhất độ dài L"` đơn điệu (mục 4), chỉ cần `O(log n)` lần kiểm tra.

**Đột phá 2 — kiểm tra `P(L)` trong `O(n)`** bằng băm cuộn: tính băm mọi cửa sổ độ dài `L` trong `O(n)`, đếm bằng `Counter`, xem có băm nào xuất hiện đúng 1 lần.

**Đột phá 3 (chính xác, không xác suất).** Sắp xếp mọi hậu tố của `nums` (mảng hậu tố) và xét **LCP** của hai hậu tố kề nhau. Một cửa sổ bắt đầu tại `i` với độ dài `L` là duy nhất ⇔ không hậu tố nào khác có tiền tố giống nó ⇔ `L >` LCP lớn nhất giữa hậu tố `i` và hậu tố bất kỳ khác ⇔ `L >` max(LCP với hàng xóm trái, LCP với hàng xóm phải) trong thứ tự sắp xếp (LCP với các hậu tố xa hơn không lớn hơn LCP với hàng xóm). Cũng cần `i + L ≤ n`. Vậy độ dài ngắn nhất bắt đầu tại `i` là `max(lcp_trái, lcp_phải) + 1` nếu `≤ n − i`; đáp án là `min` qua mọi `i`.

**Đột phá 4 — automat hậu tố.** Mỗi trạng thái đại diện nhóm các mảng con có cùng tập vị trí kết thúc; độ dài của chúng nằm trong `(len(link(v)), len(v)]`. Mảng con duy nhất ⇔ trạng thái có **đúng 1 vị trí kết thúc**; độ dài nhỏ nhất của nhóm là `len(link(v)) + 1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tìm nhị phân + băm cuộn

**Giải thích từng đoạn.**

1. Chọn mô-đun lớn `P = 2^61 − 1` và cơ số **ngẫu nhiên** `B` (tránh test chống băm cố định).
2. Tính `H` (băm tiền tố) một lần.
3. `check(L)`: tính băm của `n − L + 1` cửa sổ bằng công thức mục 3.3, dùng `Counter`, kiểm tra `1 in counter.values()`.
4. Tìm nhị phân `L` nhỏ nhất với `check(L)`; `check(n)` luôn đúng.

In [8]:
def binary_hash(nums, P=(1 << 61) - 1, B=None, seed=None):
    n = len(nums)
    if B is None:
        B = random.Random(seed).randrange(10**6, P - 1)
    H = [0] * (n + 1)
    for i, x in enumerate(nums):
        H[i + 1] = (H[i] * B + x) % P

    def check(L):
        BL = pow(B, L, P)
        cnt = Counter((H[i + L] - H[i] * BL) % P for i in range(n - L + 1))
        return 1 in cnt.values()

    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if check(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, exp in EXAMPLES:
    got = binary_hash(nums, seed=1)
    print(nums, "->", got)
    assert got == exp

[3, 3, 3] -> 3
[2, 1, 2, 3, 3] -> 1
[1, 1, 2, 2, 1] -> 2


**Độ phức tạp:** `O(n log n)` thời gian (`log n` lần kiểm tra, mỗi lần `O(n)`), `O(n)` không gian. Đáp án đúng với xác suất rất cao (xem 3.4), nhưng **không tuyệt đối**.

### 7.2. Cách 3 — Mảng hậu tố + LCP (chính xác)

**Giải thích từng đoạn.**

1. **Mảng hậu tố bằng nhân đôi (prefix doubling):** `rank[i]` là hạng của hậu tố `i` theo `k` phần tử đầu; ở bước `k` sắp xếp theo cặp `(rank[i], rank[i+k])` để có hạng theo `2k` phần tử; lặp cho tới khi mọi hạng khác nhau. Mỗi vòng `O(n log n)` (sort), có `O(log n)` vòng.
2. **LCP bằng thuật toán Kasai:** `lcp[r]` = độ dài tiền tố chung của hậu tố hạng `r−1` và `r`; duyệt `i` theo vị trí gốc, tận dụng `h` giảm không quá 1 mỗi bước ⇒ `O(n)`.
3. Với hậu tố hạng `r`, `need = max(lcp[r], lcp[r+1]) + 1`; hợp lệ nếu `need ≤ n − sa[r]` (đủ dài để tồn tại). Đáp án `min need`.

In [9]:
def suffix_array_lcp(nums):
    n = len(nums)
    comp = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
    rank = [comp[v] for v in nums]
    sa = list(range(n))
    k = 1
    while True:
        key = [rank[i] * (n + 2) + (rank[i + k] if i + k < n else 0) for i in range(n)]
        sa.sort(key=key.__getitem__)
        new_rank = [0] * n
        new_rank[sa[0]] = 1
        for j in range(1, n):
            new_rank[sa[j]] = new_rank[sa[j - 1]] + (key[sa[j]] != key[sa[j - 1]])
        rank = new_rank
        if rank[sa[-1]] == n:
            break
        k *= 2
    pos = [0] * n                           # pos[i] = hạng của hậu tố i
    for r, s in enumerate(sa):
        pos[s] = r
    lcp = [0] * (n + 1)                     # lcp[r] = LCP(hậu tố hạng r-1, hậu tố hạng r); lcp[0] = lcp[n] = 0
    h = 0
    for i in range(n):
        if pos[i] > 0:
            j = sa[pos[i] - 1]
            while i + h < n and j + h < n and nums[i + h] == nums[j + h]:
                h += 1
            lcp[pos[i]] = h
            if h:
                h -= 1
        else:
            h = 0
    return sa, lcp

def smallest_unique_sa(nums):
    n = len(nums)
    sa, lcp = suffix_array_lcp(nums)
    best = n
    for r in range(n):
        need = max(lcp[r], lcp[r + 1]) + 1
        if need <= n - sa[r]:
            best = min(best, need)
    return best

for nums, exp in EXAMPLES:
    got = smallest_unique_sa(nums)
    print(nums, "->", got)
    assert got == exp

[3, 3, 3] -> 3
[2, 1, 2, 3, 3] -> 1
[1, 1, 2, 2, 1] -> 2


**Độ phức tạp:** `O(n log² n)` thời gian (sort trong vòng nhân đôi; thực tế rất nhanh vì dừng sớm khi các hạng đã khác nhau), `O(n)` không gian. Kết quả **chính xác**.

### 7.3. Cách 4 — Automat hậu tố (suffix automaton)

**Ý tưởng.** Xây automat hậu tố của `nums` (mỗi phần tử là một "ký tự", dùng `dict` làm bảng chuyển). Mỗi trạng thái `v` có:

- `len[v]`: độ dài mảng con dài nhất trong nhóm;
- `link[v]`: liên kết hậu tố (trạng thái của hậu tố dài nhất **không** thuộc nhóm của `v`);
- `cnt[v]`: số lần các mảng con trong nhóm `v` xuất hiện (số vị trí kết thúc).

Các mảng con của nhóm `v` có độ dài `len[link[v]] + 1 .. len[v]` và đều xuất hiện `cnt[v]` lần. Vậy mảng con duy nhất ngắn nhất = `min` của `len[link[v]] + 1` qua các trạng thái có `cnt[v] = 1`.

**Cách tính `cnt`.** Khi thêm phần tử mới tạo trạng thái `cur` thì `cnt[cur] = 1` (kết thúc tại vị trí này); trạng thái bản sao (clone) có `cnt = 0`. Cuối cùng cộng dồn `cnt` từ con lên cha theo `link`, theo thứ tự `len` giảm dần.

In [10]:
def smallest_unique_sam(nums):
    mx = [0]; link = [-1]; nxt = [{}]; cnt = [0]
    last = 0
    for c in nums:
        cur = len(mx)
        mx.append(mx[last] + 1); link.append(-1); nxt.append({}); cnt.append(1)
        p = last
        while p != -1 and c not in nxt[p]:
            nxt[p][c] = cur
            p = link[p]
        if p == -1:
            link[cur] = 0
        else:
            q = nxt[p][c]
            if mx[p] + 1 == mx[q]:
                link[cur] = q
            else:                                   # tách (clone) trạng thái q
                cl = len(mx)
                mx.append(mx[p] + 1); link.append(link[q]); nxt.append(dict(nxt[q])); cnt.append(0)
                while p != -1 and nxt[p].get(c) == q:
                    nxt[p][c] = cl
                    p = link[p]
                link[q] = link[cur] = cl
        last = cur
    # cộng dồn cnt theo link, từ len lớn đến nhỏ
    for v in sorted(range(1, len(mx)), key=lambda v: -mx[v]):
        cnt[link[v]] += cnt[v]
    return min(mx[link[v]] + 1 for v in range(1, len(mx)) if cnt[v] == 1)

for nums, exp in EXAMPLES:
    got = smallest_unique_sam(nums)
    print(nums, "->", got)
    assert got == exp

[3, 3, 3] -> 3
[2, 1, 2, 3, 3] -> 1
[1, 1, 2, 2, 1] -> 2


**Độ phức tạp:** `O(n)` trạng thái (`≤ 2n`), mỗi lần thêm phần tử `O(1)` khấu hao (với `dict`), sắp xếp theo `len` `O(n log n)` (có thể dùng đếm phân loại `O(n)`). Không gian `O(n)`. Kết quả **chính xác**. Đây là lời giải ta dùng để nộp (nhanh, không xác suất).

## 8. TRACE từng bước

### 8.1. Tìm nhị phân trên `nums = [1,1,2,2,1]`

Bảng:

| bước | `lo` | `hi` | `mid` | các cửa sổ độ dài `mid` | có duy nhất? | cập nhật |
|---|---|---|---|---|---|---|
| 1 | 1 | 5 | 3 | `(1,1,2)`, `(1,2,2)`, `(2,2,1)` | có | `hi = 3` |
| 2 | 1 | 3 | 2 | `(1,1)`, `(1,2)`, `(2,2)`, `(2,1)` | có | `hi = 2` |
| 3 | 1 | 2 | 1 | `(1)×3`, `(2)×2` | không | `lo = 2` |

Kết thúc `lo = hi = 2`.

In [11]:
def trace_binary_search(nums):
    n = len(nums)
    lo, hi = 1, n
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        c = Counter(tuple(nums[i:i + mid]) for i in range(n - mid + 1))
        uniq = [w for w, v in c.items() if v == 1]
        step += 1
        print(f"bước {step}: lo={lo} hi={hi} mid={mid}  đếm={dict(c)}  duy nhất={uniq}")
        if uniq:
            hi = mid
            print(f"         có cửa sổ duy nhất -> hi = {hi}")
        else:
            lo = mid + 1
            print(f"         không có -> lo = {lo}")
    print("đáp án =", lo)
    return lo

assert trace_binary_search([1, 1, 2, 2, 1]) == 2

bước 1: lo=1 hi=5 mid=3  đếm={(1, 1, 2): 1, (1, 2, 2): 1, (2, 2, 1): 1}  duy nhất=[(1, 1, 2), (1, 2, 2), (2, 2, 1)]
         có cửa sổ duy nhất -> hi = 3
bước 2: lo=1 hi=3 mid=2  đếm={(1, 1): 1, (1, 2): 1, (2, 2): 1, (2, 1): 1}  duy nhất=[(1, 1), (1, 2), (2, 2), (2, 1)]
         có cửa sổ duy nhất -> hi = 2
bước 3: lo=1 hi=2 mid=1  đếm={(1,): 3, (2,): 2}  duy nhất=[]
         không có -> lo = 2
đáp án = 2


### 8.2. Mảng hậu tố + LCP trên `nums = [1,1,2,2,1]`

Các hậu tố: `0:[1,1,2,2,1]`, `1:[1,2,2,1]`, `2:[2,2,1]`, `3:[2,1]`, `4:[1]`. Sắp xếp tăng dần: `[1]` (4), `[1,1,2,2,1]` (0), `[1,2,2,1]` (1), `[2,1]` (3), `[2,2,1]` (2). Hãy chạy để xem LCP và `need` của từng hậu tố.

In [12]:
def trace_sa(nums):
    n = len(nums)
    sa, lcp = suffix_array_lcp(nums)
    print("mảng hậu tố sa =", sa)
    for r, s in enumerate(sa):
        print(f"hạng {r}: hậu tố bắt đầu {s} = {nums[s:]}   lcp với hạng trước = {lcp[r]}")
    best = n
    for r in range(n):
        need = max(lcp[r], lcp[r + 1]) + 1
        ok = need <= n - sa[r]
        print(f"hạng {r} (bắt đầu {sa[r]}): need = max({lcp[r]}, {lcp[r + 1]}) + 1 = {need}, còn {n - sa[r]} phần tử -> {'hợp lệ' if ok else 'quá dài, bỏ'}")
        if ok:
            best = min(best, need)
    print("đáp án =", best)
    return best

assert trace_sa([1, 1, 2, 2, 1]) == 2

mảng hậu tố sa = [4, 0, 1, 3, 2]
hạng 0: hậu tố bắt đầu 4 = [1]   lcp với hạng trước = 0
hạng 1: hậu tố bắt đầu 0 = [1, 1, 2, 2, 1]   lcp với hạng trước = 1
hạng 2: hậu tố bắt đầu 1 = [1, 2, 2, 1]   lcp với hạng trước = 1
hạng 3: hậu tố bắt đầu 3 = [2, 1]   lcp với hạng trước = 0
hạng 4: hậu tố bắt đầu 2 = [2, 2, 1]   lcp với hạng trước = 1
hạng 0 (bắt đầu 4): need = max(0, 1) + 1 = 2, còn 1 phần tử -> quá dài, bỏ
hạng 1 (bắt đầu 0): need = max(1, 1) + 1 = 2, còn 5 phần tử -> hợp lệ
hạng 2 (bắt đầu 1): need = max(1, 0) + 1 = 2, còn 4 phần tử -> hợp lệ
hạng 3 (bắt đầu 3): need = max(0, 1) + 1 = 2, còn 2 phần tử -> hợp lệ
hạng 4 (bắt đầu 2): need = max(1, 0) + 1 = 2, còn 3 phần tử -> hợp lệ
đáp án = 2


### 8.3. Xây automat hậu tố trên `nums = [1, 2, 1]`

Hãy xem từng lần `extend`: trạng thái mới `cur`, các chuyển được thêm, và khi nào phải tách (clone).

In [13]:
def trace_sam(nums):
    mx = [0]; link = [-1]; nxt = [{}]; cnt = [0]
    last = 0
    for step, c in enumerate(nums):
        cur = len(mx)
        mx.append(mx[last] + 1); link.append(-1); nxt.append({}); cnt.append(1)
        print(f"\nthêm nums[{step}] = {c}: trạng thái mới {cur} (len={mx[cur]}), last trước = {last}")
        p = last
        while p != -1 and c not in nxt[p]:
            nxt[p][c] = cur
            print(f"   trạng thái {p}: chưa có chuyển '{c}', thêm {p} --{c}--> {cur}")
            p = link[p]
        if p == -1:
            link[cur] = 0
            print(f"   chạm đáy: link[{cur}] = 0")
        else:
            q = nxt[p][c]
            if mx[p] + 1 == mx[q]:
                link[cur] = q
                print(f"   gặp {p}, q = {q}, len khớp -> link[{cur}] = {q}")
            else:
                cl = len(mx)
                mx.append(mx[p] + 1); link.append(link[q]); nxt.append(dict(nxt[q])); cnt.append(0)
                print(f"   gặp {p}, q = {q}, len không khớp -> tách clone {cl} (len={mx[cl]})")
                while p != -1 and nxt[p].get(c) == q:
                    nxt[p][c] = cl
                    p = link[p]
                link[q] = link[cur] = cl
        last = cur
    for v in sorted(range(1, len(mx)), key=lambda v: -mx[v]):
        cnt[link[v]] += cnt[v]
    print("\nkết quả: trạng thái | len | link | cnt | độ dài nhóm")
    for v in range(1, len(mx)):
        print(f"   {v:>3} | {mx[v]:>3} | {link[v]:>4} | {cnt[v]:>3} | {mx[link[v]] + 1}..{mx[v]}")
    ans = min(mx[link[v]] + 1 for v in range(1, len(mx)) if cnt[v] == 1)
    print("đáp án = min(len(link)+1 | cnt == 1) =", ans)
    return ans

assert trace_sam([1, 2, 1]) == brute([1, 2, 1]) == 1


thêm nums[0] = 1: trạng thái mới 1 (len=1), last trước = 0
   trạng thái 0: chưa có chuyển '1', thêm 0 --1--> 1
   chạm đáy: link[1] = 0

thêm nums[1] = 2: trạng thái mới 2 (len=2), last trước = 1
   trạng thái 1: chưa có chuyển '2', thêm 1 --2--> 2
   trạng thái 0: chưa có chuyển '2', thêm 0 --2--> 2
   chạm đáy: link[2] = 0

thêm nums[2] = 1: trạng thái mới 3 (len=3), last trước = 2
   trạng thái 2: chưa có chuyển '1', thêm 2 --1--> 3
   gặp 0, q = 1, len khớp -> link[3] = 1

kết quả: trạng thái | len | link | cnt | độ dài nhóm
     1 |   1 |    0 |   2 | 1..1
     2 |   2 |    0 |   1 | 1..2
     3 |   3 |    1 |   1 | 2..3
đáp án = min(len(link)+1 | cnt == 1) = 1


## 9. Code nộp LeetCode

Nộp bản **automat hậu tố** (chính xác, `O(n)` trạng thái).

In [14]:
from typing import List

class Solution:
    def smallestUniqueSubarray(self, nums: List[int]) -> int:
        mx = [0]            # độ dài mảng con dài nhất của trạng thái
        link = [-1]         # liên kết hậu tố
        nxt = [{}]          # chuyển trạng thái
        cnt = [0]           # số lần xuất hiện (số vị trí kết thúc)
        last = 0
        for c in nums:
            cur = len(mx)
            mx.append(mx[last] + 1)
            link.append(-1)
            nxt.append({})
            cnt.append(1)
            p = last
            while p != -1 and c not in nxt[p]:
                nxt[p][c] = cur
                p = link[p]
            if p == -1:
                link[cur] = 0
            else:
                q = nxt[p][c]
                if mx[p] + 1 == mx[q]:
                    link[cur] = q
                else:
                    cl = len(mx)
                    mx.append(mx[p] + 1)
                    link.append(link[q])
                    nxt.append(dict(nxt[q]))
                    cnt.append(0)
                    while p != -1 and nxt[p].get(c) == q:
                        nxt[p][c] = cl
                        p = link[p]
                    link[q] = link[cur] = cl
            last = cur
        for v in sorted(range(1, len(mx)), key=lambda v: -mx[v]):
            cnt[link[v]] += cnt[v]
        return min(mx[link[v]] + 1 for v in range(1, len(mx)) if cnt[v] == 1)

sol = Solution()
for nums, exp in EXAMPLES:
    got = sol.smallestUniqueSubarray(nums)
    print(nums, "->", got)
    assert got == exp

[3, 3, 3] -> 3
[2, 1, 2, 3, 3] -> 1
[1, 1, 2, 2, 1] -> 2


## 10. Vì sao thuật toán đúng

**Tìm nhị phân (Cách 2).** Đã chứng minh đơn điệu ở mục 4. `check(L)` đúng khi cửa sổ có băm xuất hiện đúng một lần là duy nhất thật (băm bằng nhau khi nội dung bằng nhau, nên một cửa sổ duy nhất có băm duy nhất — **trừ khi va chạm**). Va chạm có thể làm `check` sai theo một chiều: hai cửa sổ khác nhau trùng băm khiến một cửa sổ duy nhất bị coi là "xuất hiện 2 lần" ⇒ có thể bỏ sót `L`. Xác suất ≈ `n²/(2P)`; với `P = 2^61 − 1` là không đáng kể.

**Mảng hậu tố + LCP (Cách 3).** Cửa sổ `[i, i+L)` là duy nhất ⇔ không hậu tố `j ≠ i` có `nums[j:j+L] = nums[i:i+L]` ⇔ `LCP(i, j) < L` với mọi `j ≠ i` (kể cả khi `j + L > n` thì không thể khớp). Trong thứ tự sắp xếp, `LCP(i, j)` giảm dần khi `j` xa hạng của `i` (tính chất của mảng hậu tố), nên chỉ cần xét hai hàng xóm: `L > max(lcp_trái, lcp_phải)`. Thêm điều kiện cửa sổ tồn tại: `i + L ≤ n`. Độ dài nhỏ nhất tại `i` là `max(...) + 1` nếu thỏa điều kiện.

**Automat hậu tố (Cách 4).** Trong automat, mọi mảng con của `nums` thuộc đúng một trạng thái `v`; mọi mảng con của `v` có cùng tập vị trí kết thúc (nên cùng số lần xuất hiện `cnt[v]`), với độ dài trải đều từ `len[link[v]] + 1` đến `len[v]`. Một mảng con xuất hiện đúng một lần ⇔ nhóm của nó có `cnt = 1`. Độ dài nhỏ nhất trong nhóm là `len[link[v]] + 1`. `cnt` tính đúng nhờ: trạng thái tạo khi thêm phần tử thứ `t` có một vị trí kết thúc ở `t`; clone không có vị trí kết thúc riêng; cộng dồn lên `link` thêm các vị trí kết thúc của các trạng thái con (mọi mảng con có hậu tố...). Test vét cạn ở mục 12 xác nhận cả ba.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`.

| Cách | Thời gian | Không gian | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(n · ans²)` → `O(n³)` | `O(n · L)` | chỉ kiểm chứng |
| 2. Nhị phân + băm | `O(n log n)` | `O(n)` | xác suất đúng rất cao |
| 3. Mảng hậu tố + LCP | `O(n log² n)` | `O(n)` | chính xác |
| 4. Automat hậu tố | `O(n)` trạng thái + sắp xếp `O(n log n)` | `O(n)` | chính xác, nhanh nhất thực tế |

Với `n = 10^5`, Cách 4 xây xong trong khoảng `0.1–0.3` giây trong Python (xem mục 12).

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, toàn số giống nhau, toàn số khác nhau, luân phiên), đối chiếu ngẫu nhiên giữa vét cạn và các cách, và thời gian với `n = 10^5`.

In [15]:
sol = Solution()
for nums, exp in EXAMPLES:
    assert sol.smallestUniqueSubarray(nums) == exp

assert sol.smallestUniqueSubarray([7]) == 1
assert sol.smallestUniqueSubarray([5, 5, 5, 5, 5]) == 5                  # cả mảng mới duy nhất
assert sol.smallestUniqueSubarray([1, 2, 3, 4]) == 1                     # mọi phần tử duy nhất
assert sol.smallestUniqueSubarray([1, 2, 1, 2]) == 2                     # (1,2) xuất hiện 2 lần nhưng (2,1) chỉ 1 lần
assert sol.smallestUniqueSubarray([1, 2, 1, 2]) == brute([1, 2, 1, 2])
assert sol.smallestUniqueSubarray([1, 1, 2, 1, 1]) == 1                  # 2 duy nhất
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [16]:
rng = random.Random(3934)
for _ in range(500):
    n = rng.randint(1, 14)
    alpha = rng.choice([1, 2, 2, 3, 5, 100])
    nums = [rng.randint(1, alpha) for _ in range(n)]
    exp = brute(nums)
    assert binary_hash(nums, seed=rng.randint(1, 10**9)) == exp, nums
    assert smallest_unique_sa(nums) == exp, nums
    assert smallest_unique_sam(nums) == exp, nums
    assert sol.smallestUniqueSubarray(nums) == exp, nums
print("500 test nhỏ: 5 cách khớp nhau")

for _ in range(100):
    n = rng.randint(20, 200)
    alpha = rng.choice([1, 2, 3, 4])
    nums = [rng.randint(1, alpha) for _ in range(n)]
    exp = brute(nums)
    assert smallest_unique_sa(nums) == smallest_unique_sam(nums) == sol.smallestUniqueSubarray(nums) == binary_hash(nums, seed=7) == exp
print("100 test vừa (bảng chữ nhỏ, nhiều lặp) khớp")

500 test nhỏ: 5 cách khớp nhau


100 test vừa (bảng chữ nhỏ, nhiều lặp) khớp


In [17]:
# Thời gian ở giới hạn n = 1e5 với các dạng khó
rng = random.Random(1)
n = 10**5
cases = {
    "ngẫu nhiên [1, 1e5]": [rng.randint(1, 10**5) for _ in range(n)],
    "ngẫu nhiên {1,2}": [rng.randint(1, 2) for _ in range(n)],
    "toàn số 1": [1] * n,
    "luân phiên 1,2,1,2": [1 + i % 2 for i in range(n)],
    "chu kỳ 1000": [1 + i % 1000 for i in range(n)],
}
for name, arr in cases.items():
    t0 = time.time(); a = sol.smallestUniqueSubarray(arr); t1 = time.time() - t0
    t0 = time.time(); b = binary_hash(arr, seed=3); t2 = time.time() - t0
    t0 = time.time(); c = smallest_unique_sa(arr); t3 = time.time() - t0
    assert a == b == c
    print(f"{name:22s} -> {a:6d} | SAM {t1:.2f}s  nhị phân+băm {t2:.2f}s  SA+LCP {t3:.2f}s")

ngẫu nhiên [1, 1e5]    ->      1 | SAM 0.15s  nhị phân+băm 0.52s  SA+LCP 0.15s


ngẫu nhiên {1,2}       ->     14 | SAM 0.26s  nhị phân+băm 0.64s  SA+LCP 0.26s


toàn số 1              -> 100000 | SAM 0.06s  nhị phân+băm 0.05s  SA+LCP 0.40s


luân phiên 1,2,1,2     ->  99998 | SAM 0.06s  nhị phân+băm 0.05s  SA+LCP 0.41s


chu kỳ 1000            ->  98002 | SAM 0.06s  nhị phân+băm 0.05s  SA+LCP 0.58s


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét phần tử đơn (độ dài 1)

**Code sai:** trả về `1` nếu có số xuất hiện đúng một lần, ngược lại `len(nums)`.
**Vì sao sai:** có thể có mảng con duy nhất độ dài `2, 3, ...` nhỏ hơn `n`.
**Code đúng:** xét mọi độ dài (tìm nhị phân/automat).

In [18]:
def wrong_only_singletons(nums):
    c = Counter(nums)
    return 1 if 1 in c.values() else len(nums)          # SAI

print("sai :", wrong_only_singletons([1, 1, 2, 2, 1]), " đúng:", Solution().smallestUniqueSubarray([1, 1, 2, 2, 1]))

sai : 5  đúng: 2


### Lỗi 2 — Đếm lần xuất hiện không chồng lấn

**Code sai:** dùng `str.count` (không chồng lấn) hoặc nhảy cửa sổ `L` bước.
**Vì sao sai:** đề coi hai mảng con ở vị trí khác nhau là khác nhau, kể cả chồng lấn: `[3,3]` xuất hiện **2 lần** trong `[3,3,3]`.
**Code đúng:** đếm mọi cửa sổ bắt đầu từ `0..n−L`.

In [19]:
s = "333"
print("'33' xuất hiện (str.count, không chồng lấn):", s.count("33"), " | thực tế cửa sổ:", sum(1 for i in range(2) if s[i:i + 2] == "33"))

def wrong_nonoverlap(nums):
    n = len(nums)
    for L in range(1, n + 1):
        seen = {}
        for i in range(0, n - L + 1, L):                # SAI: nhảy L bước
            seen[tuple(nums[i:i + L])] = seen.get(tuple(nums[i:i + L]), 0) + 1
        if 1 in seen.values():
            return L
    return n

print("sai :", wrong_nonoverlap([3, 3, 3]), " đúng:", Solution().smallestUniqueSubarray([3, 3, 3]))

'33' xuất hiện (str.count, không chồng lấn): 1  | thực tế cửa sổ: 2
sai : 2  đúng: 3


### Lỗi 3 — Băm với mô-đun quá nhỏ (va chạm)

**Code sai:** `mod = 101` (hoặc `10^9+7` với `n` lớn) và coi băm bằng nhau là mảng con bằng nhau.
**Vì sao sai:** hai cửa sổ khác nhau có thể trùng băm → cửa sổ duy nhất bị coi là trùng → đáp án sai.
**Code đúng:** mô-đun lớn (`2^61 − 1`) + cơ số ngẫu nhiên, hoặc dùng cấu trúc chính xác (suffix array/automaton).

In [20]:
rng = random.Random(9)
bad = None
for _ in range(2000):
    n = rng.randint(30, 60)
    nums = [rng.randint(1, 4) for _ in range(n)]
    B_ = rng.randint(2, 100)
    if binary_hash(nums, P=101, B=B_) != brute(nums):
        bad = (nums, B_)
        break
print("tìm thấy phản ví dụ cho mô-đun 101:", bad is not None)
nums, B_ = bad
print("sai :", binary_hash(nums, P=101, B=B_), " đúng:", brute(nums), " (n =", len(nums), ", B =", B_, ")")

tìm thấy phản ví dụ cho mô-đun 101: True
sai : 3  đúng: 2  (n = 47 , B = 34 )


### Lỗi 4 — Khóa `list` trong `dict`/`Counter`

`Counter(nums[i:i+L] for ...)` gây `TypeError: unhashable type: 'list'`. Dùng `tuple(...)` (như `has_unique_window_exact`).

### Lỗi 5 — Vét cạn `O(n³)` cho `n = 10^5`

Mỗi tuple dài `L` cần `O(L)` để băm. Với mảng toàn số giống nhau (`ans = n`) thì `Σ_L (n−L)·L ≈ n³/6`. Dưới đây so thời gian vét cạn và automat ở `n` vừa phải.

In [21]:
for n in (100, 300, 600):
    arr = [1] * n
    t0 = time.time(); a = brute(arr); t1 = time.time() - t0
    t0 = time.time(); b = Solution().smallestUniqueSubarray(arr); t2 = time.time() - t0
    assert a == b == n
    print(f"n={n:4d}: vét cạn {t1:.3f}s | automat {t2:.4f}s")

n= 100: vét cạn 0.002s | automat 0.0002s
n= 300: vét cạn 0.037s | automat 0.0002s


n= 600: vét cạn 0.271s | automat 0.0004s


## 14. Mẹo & nhận dạng pattern

- **"Độ dài nhỏ nhất sao cho tồn tại..."** + tính đơn điệu theo độ dài → **tìm nhị phân trên đáp án**; luôn chứng minh đơn điệu bằng "mở rộng cửa sổ".
- **So sánh nhiều chuỗi con** → **băm cuộn** (`O(1)` mỗi cửa sổ). Dùng mô-đun lớn + cơ số ngẫu nhiên; cân nhắc băm kép khi lo ngại.
- **Mảng con xuất hiện bao nhiêu lần / duy nhất / lặp lại dài nhất** → nghĩ ngay đến **mảng hậu tố + LCP** hoặc **automat hậu tố**.
- Ghi nhớ hai công thức: *độ dài ngắn nhất để hậu tố `i` duy nhất = `max(LCP hàng xóm) + 1`*; *nhóm trạng thái automat có `cnt = 1` cho đáp án `len(link) + 1`*.
- Với bảng chữ là số nguyên lớn (≤ `10^5`), dùng `dict` làm chuyển trạng thái của automat; hoặc nén giá trị rồi dùng mảng.

## 15. Biến thể

1. **Mảng con lặp lại dài nhất** (xuất hiện ≥ 2 lần): `max` của `len[v]` qua trạng thái có `cnt ≥ 2`; hoặc `max(lcp)` từ mảng hậu tố.
2. **Đếm số mảng con phân biệt:** `Σ (len[v] − len[link[v]])` qua mọi trạng thái.
3. **Mảng con ngắn nhất xuất hiện đúng `k` lần:** `min(len[link[v]] + 1)` qua trạng thái có `cnt[v] = k`.
4. **Phiên bản chuỗi (string) thay vì mảng:** cùng thuật toán, ký tự thay cho số.
5. **Đếm số mảng con duy nhất:** `Σ` qua trạng thái `cnt = 1` của `len[v] − len[link[v]]`.

In [22]:
# Biến thể 1, 2, 5 trên cùng một automat
def sam_stats(nums):
    mx = [0]; link = [-1]; nxt = [{}]; cnt = [0]; last = 0
    for c in nums:
        cur = len(mx); mx.append(mx[last] + 1); link.append(-1); nxt.append({}); cnt.append(1)
        p = last
        while p != -1 and c not in nxt[p]:
            nxt[p][c] = cur; p = link[p]
        if p == -1:
            link[cur] = 0
        else:
            q = nxt[p][c]
            if mx[p] + 1 == mx[q]:
                link[cur] = q
            else:
                cl = len(mx); mx.append(mx[p] + 1); link.append(link[q]); nxt.append(dict(nxt[q])); cnt.append(0)
                while p != -1 and nxt[p].get(c) == q:
                    nxt[p][c] = cl; p = link[p]
                link[q] = link[cur] = cl
        last = cur
    for v in sorted(range(1, len(mx)), key=lambda v: -mx[v]):
        cnt[link[v]] += cnt[v]
    distinct = sum(mx[v] - mx[link[v]] for v in range(1, len(mx)))
    unique = sum(mx[v] - mx[link[v]] for v in range(1, len(mx)) if cnt[v] == 1)
    longest_rep = max([mx[v] for v in range(1, len(mx)) if cnt[v] >= 2], default=0)
    return distinct, unique, longest_rep

nums = [1, 1, 2, 2, 1]
distinct, unique, longest_rep = sam_stats(nums)
# đối chiếu vét cạn
subs = Counter(tuple(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1))
assert distinct == len(subs)
assert unique == sum(1 for v in subs.values() if v == 1)
assert longest_rep == max([len(k) for k, v in subs.items() if v >= 2], default=0)
print("số mảng con phân biệt:", distinct, "| số mảng con duy nhất:", unique, "| dài nhất lặp lại:", longest_rep)

số mảng con phân biệt: 12 | số mảng con duy nhất: 10 | dài nhất lặp lại: 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử từng `L`, `Counter` các tuple | `O(n³)` | `O(n²)` | kiểm chứng, `n ≤ 14` |
| 2. Nhị phân + băm cuộn | đơn điệu + băm `O(1)` mỗi cửa sổ | `O(n log n)` | `O(n)` | nhanh để viết; chấp nhận xác suất rất nhỏ |
| 3. Mảng hậu tố + LCP | `need = max(LCP hàng xóm) + 1` | `O(n log² n)` | `O(n)` | cần đáp án chính xác, nhiều truy vấn chuỗi |
| 4. Automat hậu tố | trạng thái `cnt = 1`, `len(link) + 1` | `O(n)` (+ sort) | `O(n)` | **lời giải nộp**; chính xác, nhanh |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `has_unique_window_exact([1, 2, 1, 2, 1], 2)` bằng gì?

**Bài 2 (tính tay).** `nums = [4, 4, 4, 5]`. Đáp án của bài là gì và mảng con duy nhất đó là gì?

**Bài 3 (điền TODO).** Hoàn thiện `my_window_hashes(nums, L, B, P)` trả về danh sách băm các cửa sổ độ dài `L` bằng băm cuộn (không tạo tuple).

**Bài 4 (tự viết lại).** Viết `my_smallest_unique(nums)` bằng tìm nhị phân + `Counter` các tuple (chính xác, chậm hơn băm).

**Bài 5 (biến thể).** Viết `longest_repeated_subarray(nums)`: độ dài lớn nhất của mảng con xuất hiện **ít nhất 2 lần** (dùng tìm nhị phân + tuple; mục 15 cho thấy đáp án có thể lấy từ automat).

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_window_hashes(nums, L, B, P):
    # TODO 1: H = băm tiền tố, H[i+1] = (H[i]*B + nums[i]) % P
    # TODO 2: BL = pow(B, L, P)
    # TODO 3: trả về [(H[i+L] - H[i]*BL) % P for i in range(n - L + 1)]
    pass

# Bài 4 — TODO
def my_smallest_unique(nums):
    # TODO: tìm nhị phân L trong [1, n]; check(L) = có tuple xuất hiện đúng 1 lần
    pass

# Bài 5 — TODO
def longest_repeated_subarray(nums):
    # TODO: P(L) = có cửa sổ độ dài L xuất hiện >= 2 lần (đơn điệu giảm theo L); tìm L lớn nhất thỏa
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê các cửa sổ độ dài 2 của `[1,2,1,2,1]`.
- Mức 2: `(1,2)`, `(2,1)`, `(1,2)`, `(2,1)`.
- Mức 3: mỗi cái xuất hiện 2 lần → `False`.

**Bài 2.**
- Mức 1: độ dài 1: `4` ba lần, `5` một lần.
- Mức 2: `5` duy nhất.
- Mức 3: đáp án `1`, mảng con `[5]`.

**Bài 3.**
- Mức 1: viết `H` như ở mục 3.3.
- Mức 2: công thức cửa sổ `(H[i+L] - H[i]*B^L) % P`.
- Mức 3: dùng `pow(B, L, P)` một lần.

**Bài 4.**
- Mức 1: tái dùng `first_true`.
- Mức 2: `Counter(tuple(...))`.
- Mức 3: `check(L) = 1 in counter.values()`.

**Bài 5.**
- Mức 1: nếu một cửa sổ dài `L` lặp lại thì cửa sổ ngắn hơn (cắt bớt đầu) cũng lặp lại.
- Mức 2: `P(L)` giảm dần (đúng rồi sai); tìm `L` lớn nhất đúng.
- Mức 3: nhị phân với `mid = (lo + hi + 1) // 2`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert has_unique_window_exact([1, 2, 1, 2, 1], 2) is False
# Bài 2
assert Solution().smallestUniqueSubarray([4, 4, 4, 5]) == 1

# Bài 3
def my_window_hashes_ans(nums, L, B, P):
    n = len(nums)
    H = [0] * (n + 1)
    for i, x in enumerate(nums):
        H[i + 1] = (H[i] * B + x) % P
    BL = pow(B, L, P)
    return [(H[i + L] - H[i] * BL) % P for i in range(n - L + 1)]

nums = [3, 1, 4, 1, 5, 9, 2, 6]
for L in range(1, len(nums) + 1):
    hs = my_window_hashes_ans(nums, L, 911382323, (1 << 61) - 1)
    direct = []
    for i in range(len(nums) - L + 1):
        h = 0
        for x in nums[i:i + L]:
            h = (h * 911382323 + x) % ((1 << 61) - 1)
        direct.append(h)
    assert hs == direct
print("Bài 1–3 OK")

Bài 1–3 OK


In [25]:
# Bài 4
def my_smallest_unique_ans(nums):
    n = len(nums)
    def check(L):
        c = Counter(tuple(nums[i:i + L]) for i in range(n - L + 1))
        return 1 in c.values()
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if check(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

rng = random.Random(17)
for _ in range(200):
    a = [rng.randint(1, rng.choice([2, 3, 6])) for _ in range(rng.randint(1, 12))]
    assert my_smallest_unique_ans(a) == brute(a)

# Bài 5
def longest_repeated_subarray_ans(nums):
    n = len(nums)
    def rep(L):
        c = Counter(tuple(nums[i:i + L]) for i in range(n - L + 1))
        return any(v >= 2 for v in c.values())
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if rep(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for _ in range(200):
    a = [rng.randint(1, rng.choice([2, 3, 6])) for _ in range(rng.randint(1, 12))]
    assert longest_repeated_subarray_ans(a) == sam_stats(a)[2], a
assert longest_repeated_subarray_ans([1, 1, 1]) == 2
assert longest_repeated_subarray_ans([1, 2, 3]) == 0
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tìm nhị phân + băm):**

```python
P = 2**61 - 1; B = random base
H[i+1] = (H[i]*B + nums[i]) % P
check(L): Counter((H[i+L] - H[i]*pow(B, L, P)) % P for i in range(n - L + 1)) có giá trị 1?
lo, hi = 1, n  ->  while lo < hi: mid = (lo+hi)//2; check(mid) ? hi = mid : lo = mid + 1
```

**Template (automat hậu tố):**

```python
xây SAM (mx, link, nxt dict, cnt: cur=1, clone=0)
cộng dồn cnt theo link, len giảm dần
return min(mx[link[v]] + 1 for v ≠ gốc nếu cnt[v] == 1)
```

**Checklist trước khi submit:**

- [ ] Hiểu và nêu được lý do **đơn điệu** (mở rộng cửa sổ duy nhất vẫn duy nhất; `L = n` luôn duy nhất).
- [ ] Các lần xuất hiện **chồng lấn đều được tính** (cửa sổ bắt đầu ở mọi `0..n−L`).
- [ ] Băm: mô-đun lớn (`2^61 − 1`), cơ số ngẫu nhiên, công thức `(H[i+L] − H[i]·B^L) % P`.
- [ ] Automat: clone có `cnt = 0`, trạng thái mới có `cnt = 1`; cộng dồn theo `link` với `len` giảm dần.
- [ ] Không dùng `list` làm khóa; dùng `tuple`.
- [ ] Đã đối chiếu vét cạn trên mảng nhỏ với bảng chữ nhỏ (nhiều lặp).
- [ ] Chạy thử `n = 10^5` với mảng toàn số giống nhau, luân phiên, chu kỳ, ngẫu nhiên.